In [1]:
import numpy as np
import pandas as pd
import joblib
import random
from sklearn import metrics
from sklearn import model_selection
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
import os

# Determining important metabolites for GEM-CM simulations

Exchange reaction fluxes establish metabolite 'availability' in the simulated growth media. Exchange reactions important to random forest regression performance are interpreted to mean that the transported metabolite is crucial for determining when PA14 has viable or non-viable growth. To emphasize exchange reactions important for this viable/non-viable distinction, regression is performed using the LOWGROWTH constraint dataset, which has PA14_Biomass flux at moderately low levels (0 - 0.3 hr-1).

Sample weighting is also designed to emphasize data points underneath the PA14_Biomass median

**Metabolite importance for predicting the training dataset is interpreted by average rank order**.

In [2]:
def weight_calc(arr1, val_thresh, max_thresh):
    #---Conversion to np.array---#
    biom_val=arr1[arr1.columns[0]].to_numpy() # 'PA14_Biomass'
    pyoc_val=arr1[arr1.columns[1]].to_numpy() # 'DM_cpd01206_c'
    #---Scalar value assignment---#
    gwth_bins=np.linspace(0.00001,0.05*(int(max_thresh/0.05)+1),(int(max_thresh/0.05)+1)+1)
    wt_assign={}
    n_count=np.histogram(biom_val, bins=gwth_bins)[0]
    for x in range(len(gwth_bins)):
        if gwth_bins[x] < val_thresh and n_count[x]>0:
            wt_assign[gwth_bins[x]]=(min(round(val_thresh/gwth_bins[x],1),10)) #bins are 1 less in length than n_count but the end should exceed val_thresh
        else:
            wt_assign[gwth_bins[x]]=1.0
    #---Sample weighting determination---#
    #---Additional weighting bump when pyoc is produced---#
    weight_final=[]; wt_keys=list(wt_assign.keys())
    for i in range(len(biom_val)):
        val_check=0
        if pyoc_val[i]>0.001:
            pyoc_bump=min(int(pyoc_val[i]/0.05),2)
        elif pyoc_val[i]<0.001 and biom_val[i]<0.1: #focus on when it stops making pyoc
            pyoc_bump=4.0
        else:
            pyoc_bump=0.0
        for j in range(len(wt_keys)-1):
            if wt_keys[j] < biom_val[i] <= wt_keys[j+1]:
                weight_final.append(wt_assign[wt_keys[j]]+pyoc_bump)
                val_check+=1
                break
            else:
                continue
        if val_check==0:
            print(biom_val[i], wt_assign[wt_keys[0]]+pyoc_bump)
            weight_final.append(wt_assign[wt_keys[0]]+pyoc_bump)
    return np.array(weight_final)

class RFRegression_FeatureEval:
    def __init__(self, df_training, nTargets, gwth_cutoff):
        # gwth_cutoff a list (length 2) for biomass flux boundaries
        labels=list(df_training.columns)
        self.feature_x=labels[:-1*int(nTargets)]
        self.xset=df_training.loc[:, labels[:-1*int(nTargets)]]
        y1=df_training.loc[:,labels[-1*int(nTargets):]]
        y1_valset=[len(set(y1[y1.columns[j]])) for j in range(y1.shape[1])]
        y_target=pd.DataFrame()
        for j in range(y1.shape[1]):
            y_target[y1.columns[j]]=y1[y1.columns[j]].to_numpy()    
        self.yset=y_target
        self.lowmux=df_training[df_training["PA14_Biomass"] <= gwth_cutoff[0]].loc[:, labels[:-1*int(nTargets)]]
        self.lowmuy=y1[y1["PA14_Biomass"] <= gwth_cutoff[0]].loc[:, list(y1.columns)]
        self.highmux=df_training[df_training["PA14_Biomass"] > gwth_cutoff[1]].loc[:, labels[:-1*int(nTargets)]]
        self.highmuy=y1[y1["PA14_Biomass"] > gwth_cutoff[1]].loc[:, list(y1.columns)]
        
    def forest_test(self,X0,y0,rand_states, rf_dict):
        n_perform=[] # tracking root mean square error of biomass flux
        rf_bag=[]
        df_feature=pd.DataFrame()
        for i in range(len(rand_states)):
            trainX, testX, trainY, testY=train_test_split(X0, y0, test_size=0.3,
                                                         random_state=rand_states[i])
            rf_n=RandomForestRegressor(**rf_dict[rand_states[i]])
            rf_feat=pd.DataFrame()
            if y0.shape[1]==1:
                biom_weight=weight_calc(np.ravel(trainY),trainY["PA14_Biomass"].median(),trainY["PA14_Biomass"].max())
                rf_n.fit(trainX, np.ravel(trainY),sample_weight=biom_weight)
                y1_test=rf_n.predict(testX)
                rmse_test=round(metrics.mean_squared_error(y_true=testY[:].to_numpy(), y_pred=y1_test)**0.5,4)
                
            else:
                biom_weight=weight_calc(trainY, trainY["PA14_Biomass"].median(), trainY["PA14_Biomass"].max())
                rf_n.fit(trainX,trainY,sample_weight=biom_weight)
                y1_test=rf_n.predict(testX)
                rmse_test=round(metrics.mean_squared_error(y_true=testY.to_numpy()[:,0], y_pred=y1_test[:,0])**0.5,4)
            
            rf_feat["Exchange"]=rf_n.feature_names_in_
            rf_feat["Decision Importance"]=rf_n.feature_importances_
            feat_sorted=rf_feat.sort_values(by="Decision Importance", ascending=False)
            explic_ord=[]
            for j in range(feat_sorted.shape[0]):
                explic_ord.append(feat_sorted.iloc[j]["Exchange"])
                
            df_feature[f'RandSeed_{rand_states[i]}']=explic_ord
            n_perform.append(rmse_test)
            rf_bag.append(rf_n.oob_score_)
            
        return df_feature, n_perform, rf_bag

In [ ]:
file_path='C:/Users/msk4u/Ford Lab/FBA_neuralnet/'
dataset=pd.read_csv(file_path+'FBACalc_ObjC2_Step03_PredictorREDUCE_del metH.csv')
data_split=rfRegression_stage1(dataset, 2, [0.31, 0.9])
x1=data_split.lowmux
y1=data_split.lowmuy

#---Stoicastic iterations---#
rng=np.random.default_rng(seed=27)
rand_states=rng.integers(23,3242,size=50, endpoint=True)
N_estim=rng.integers(100, 500, size=50, endpoint=True)
leaves=rng.integers(10,20,size=50, endpoint=True)
ensem_param={}
for j in range(len(rand_states)):
    ensem_param[rand_states[j]]={
            'n_estimators' : N_estim[j],
            'min_samples_leaf': leaves[j],
            'oob_score': True,
            'random_state': rand_states[j]
        }

### Pre-treatment: Removing training features with small importance to regression.

To help identify the 3 most important features from a large initial pool, one can first prune out training features with negligible importance.

In [ ]:
def rForest_trim(x_arr,y_arr, feat_out, rand_num):
    trainX, testX, trainY, testY=train_test_split(x_arr,y_arr,test_size=0.3)
    rf1=RandomForestRegressor(n_estimators=500, min_samples_leaf=10, 
                              oob_score=True, random_state=rand_num)
    if y_arr.shape[1]==1:
        biom_weight=weight_calc(np.ravel(trainY), trainY["PA14_Biomass"].median(), trainY["PA14_Biomass"].max())
        rf1.fit(trainX,np.ravel(trainY),sample_weight=biom_weight)
    else:
        biom_weight=weight_calc(trainY, trainY["PA14_Biomass"].median(), trainY["PA14_Biomass"].max())
        rf1.fit(trainX,trainY,sample_weight=biom_weight)
        
    rf_feat=pd.DataFrame()
    rf_feat["Exchange"]=rf1.feature_names_in_
    rf_feat["Decision Importance"]=rf1.feature_importances_
    feat_sorted=rf_feat.sort_values(by="Decision Importance", ascending=True)
    low_exchange=[]
    for i in range(int(feat_out)):
        low_exchange.append(feat_sorted.iloc[i]["Exchange"])
    return low_exchange, feat_sorted

def abundance_count(label_list):
    n_count={}
    for x in range(len(label_list)):
        for y in label_list[x]:
            if y in list(n_count.keys()):
                n_count[y]+=1
            else:
                n_count[y]=1
    n_group=list(n_count.keys())
    df_count=pd.DataFrame(columns=["Exchange","n-Observations"])
    for n in range(len(n_group)):
        df_count.loc[n]=[n_group[n], n_count[n_group[n]]]
    return df_count

def forest_prune(xdata, ydata,n_drop,rand_states):
    prune_pool=[]
    prune_final=[]
    forest_decision=[]
    ind_pull=np.arange(0,24)
    random.shuffle(ind_pull)
    for i in range(25):
        exchange_drop, full_sort=rForest_trim(xdata, ydata, n_drop, rand_states[i])
        prune_pool.append(exchange_drop)
        forest_decision.append(full_sort)
    count_hit=abundance_count(prune_pool)
    for x in range(count_hit.shape[0]):
        if count_hit.loc[x,"n-Observations"] > 17:
            prune_final.append(count_hit.loc[x,"Exchange"])
    print(f"Exchanges for removal: {prune_final}")
    print('Decision importances (random of 25 runs')
    print(forest_decision[ind_pull[0]])
    return prune_final

In [ ]:
sweep1=forest_prune(x1, y_target, 10, rand_states) #dropping at least 10 predictors
print(len(sweep1))

exchange_ordered=data_split.feature_x
exchange_pass=[x for x in exchange_ordered]
for ex in sweep1:
    exchange_pass.remove(ex)

### n = 50 random forest regression iterations to determine consistently important features

In [ ]:
# exchange_pass=[x for x in data_split.feature_x]# Uncomment if pre-trimming is not exercised
x2=x1.loc[:,exchange_pass]
stoic_run, scores, oob_n = data_split.forest_test(x2, y1, rand_states, ensem_param)

avg_count={x : 0 for x in exchange_pass}

for i in range(stoic_run.shape[0]):
    stoic_count=i+1
    for j in stoic_run.iloc[i].to_numpy():
        avg_count[j] += stoic_count
        
comp_feature=pd.DataFrame()
comp_feature['Exchange']=exchange_pass
comp_feature['Average Placement']=[avg_count[x]/50 for x in exchange_pass]

col2=list(ensem_param[rand_states[0]].keys())
stoic_condition=pd.DataFrame(columns=col2)
for j in range(len(ensem_param)):
    n_row=[]
    for k in col2:
        n_row.append(ensem_param[rand_states[j]][k])
    stoic_condition.loc[len(stoic_condition)]=n_row
stoic_condition['rMSE_Biomass']=scores
comp_feature

In [ ]:
stoic_run.to_csv('RFStage1_ObjC2_Step03_del metH final.csv', index=False)
stoic_condition.to_csv('RFstoicset_ObjC2_Step03_del metH final.csv', index=False)

# Training random forest regression (ensemble) for GEM-CM 

Regression is performed using the HIGHGROWTH constraint dataset, which has PA14_Biomass spanning from 0 - 0.9 hr-1. Only the 3 exchange reactions with high feature importance are included as training inputs. Training targets remain PA14_Biomass and DM_cpd01206_c (pyocyanin synthesis).

Sample weighting emphasizes data points underneath the PA14_Biomass median. 3 layers of gridsearching performed using GridSearchCV to narrow 'criterion', number of decision trees, and minimum samples per decision tree leaf + minimum samples for a decision tree split. 


In [ ]:
# os.chdir('C:/Users/msk4u/Ford Lab/FBA_neuralnet')
exchange_final=['EX_cpd00051_e','EX_cpd00060_e','EX_cpd00027_e']
dataset2=pd.read_csv('FBACalc_ObjC2_Step02_FinalEnsem_WT.csv')
data_split=rfRegression_stage1(dataset2, 2, [0.25, 0.5])
xtrim=data_split.xset.loc[:,exchange_final]
y_tot=data_split.yset

ytot_valset=[len(set(y_tot[y_tot.columns[j]])) for j in range(y_tot.shape[1])]
y_target=pd.DataFrame()
for j in range(y_tot.shape[1]):
    y_target[y_tot.columns[j]]=y_tot[y_tot.columns[j]].to_numpy()
trainX, testX, trainY, testY=train_test_split(xtrim,y_target,
                                              test_size=0.2,random_state=91)

In [ ]:
nleaf_search=[int(x) for x in np.linspace(11,47,20)]
estim_search=[int(x) for x in np.linspace(100,500,6)]
param_cons={'random_state': 23}
print(f'Decision tree leaf size: {nleaf_search}')
print(f'Estimator search:{estim_search}')

biom_weight=weight_calc(trainY, y_target["PA14_Biomass"].quantile(0.5), y_target["PA14_Biomass"].max())

In [ ]:
GSCV_1={
    'criterion': ['squared_error','friedman_mse'],
    'min_samples_leaf': np.array(nleaf_search),
    'n_estimators': np.array(estim_search),
}

GSCV_rep=GridSearchCV(RandomForestRegressor(**param_cons),GSCV_1,scoring='r2',n_jobs=25)
gridsearch1=GSCV_rep.fit(trainX, trainY, sample_weight=np.array(biom_weight))
gridCV_res=pd.DataFrame(gridsearch1.cv_results_).sort_values(by="rank_test_score")
gridCV_res.head(10)

In [ ]:
param_cons2={'criterion': 'squared_error',
             'random_state': 23} # adjust parameters as gridsearching progresses

nleaf_search=[int(x) for x in np.linspace(5,37,7)]
nsplit_search=[int(x) for x in np.linspace(41,92,6)] 
print(f'Decision tree split point: {nsplit_search}')

estim_search=[int(x) for x in np.linspace(100,500,9)]
print(estim_search)

In [ ]:
GSCV_2={
    'min_samples_split': np.array(nsplit_search),
    'min_samples_leaf': np.array(nleaf_search),
    'n_estimators': np.array(estim_search),
}

GSCV_rep=GridSearchCV(RandomForestRegressor(**param_cons2),GSCV_2,scoring='r2',n_jobs=25)
# gridsearch2=GSCV_rep.fit(trainX, np.ravel(trainY))
gridsearch2=GSCV_rep.fit(trainX, trainY, sample_weight=biom_weight)
gridCV_res=pd.DataFrame(gridsearch2.cv_results_).sort_values(by="rank_test_score")
print(gridCV_res['mean_test_score'].iloc[0])
gridCV_res.head(12)

In [ ]:
param_cons3={'n_estimators': 500,
             'criterion' : 'squared_error',
             'random_state': 23}
nleaf_search=[int(x) for x in np.linspace(5,7,3)]
nsplit_search=[int(x) for x in np.linspace(45,75,31)] #adjust upper bound of split
print(f'Decision tree leaf size: {nleaf_search}')
print(f'Decision tree split point: {nsplit_search}')

In [ ]:
GSCV_3={
    'min_samples_leaf': np.array(nleaf_search),
    'min_samples_split': np.array(nsplit_search)}

GSCV_rep=GridSearchCV(RandomForestRegressor(**param_cons3),GSCV_3,scoring='r2',n_jobs=25)
gridsearch3=GSCV_rep.fit(trainX, trainY, sample_weight=biom_weight)
gridCV_res=pd.DataFrame(gridsearch3.cv_results_).sort_values(by="rank_test_score")
print(gridCV_res['mean_test_score'].iloc[0])
gridCV_res.head(8)

In [ ]:
#----FINAL ENSEMBLE----#
rf_CV=RandomForestRegressor(n_estimators=500,criterion='squared_error', oob_score=True,
                            min_samples_split=6, min_samples_leaf=70,random_state=23)

rf_CV.fit(trainX, trainY, sample_weight=biom_weight)
y1_test=rf_CV.predict(testX)
df_perform=pd.DataFrame(columns=["MAE","rMSE","r2_score"])
if y_target.shape[1]==1:
    for j in range(y_target.shape[1]):
        mae_test=round(metrics.mean_absolute_error(y_true=testY, y_pred=y1_test),4)
        rmse_test=round(metrics.mean_squared_error(y_true=testY, y_pred=y1_test)**0.5,4)
        r2_test=round(metrics.r2_score(y_true=testY, y_pred=y1_test),3)
        df_perform.loc[j]=[mae_test,rmse_test,r2_test]
else:
    for j in range(y_target.shape[1]):
        mae_test=round(metrics.mean_absolute_error(y_true=testY.to_numpy()[:,j], y_pred=y1_test[:,j]),4)
        rmse_test=round(metrics.mean_squared_error(y_true=testY.to_numpy()[:,j], y_pred=y1_test[:,j])**0.5,4)
        r2_test=round(metrics.r2_score(y_true=testY.to_numpy()[:,j], y_pred=y1_test[:,j]),3)
        df_perform.loc[j]=[mae_test,rmse_test,r2_test]
df_perform.rename(index={0: "Biomass Flux", 1: "Pyoc Flux"},inplace=True)
df_perform

In [ ]:
df_parity=pd.DataFrame()
df_parity['FBA_Biom']=testY['PA14_Biomass'].to_numpy()
df_parity['RF_Biom']=y1_test[:,0]
df_parity['FBA_Pyoc']=testY['DM_cpd01206_c'].to_numpy()
df_parity['RF_Pyoc']=y1_test[:,1]

df_parity.to_csv('ParitySet_Step02_GlucArgMet_WT.csv',index=False)

file_name = 'Step02_GlucArgMet_WT_V2.joblib' 
joblib.dump(rf_CV, file_name)